## Spell Correct for Mendeley and Kaggle partially based on BOWs
Results:

**TFIDF Comparison**:


Full dataset accuracy:   0.7711 -> 0.7705


Subset (N=2000) accuracy: 0.7780 -> 0.7830


Difference: + 0.0069 -> +0.0125

**word2vec Comparison**:


Full dataset accuracy:   0.7567 -> 0.7561


Subset (N=2000) accuracy: 0.7570 -> 0.7575


Difference: +0.0003 -> +0.0014


The following way to spellcheck did not improve accuracy...

In [15]:
import json
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
import autocorrect
import re

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
# ============ Open BOW and datasets ============
print("Loading BOWs and datasets...")
with open('tokens/total_BOW_kaggle.json', 'r') as f:
    kBOW = json.load(f)
with open('tokens/total_BOW_mendeley.json', 'r') as f:
    mBOW = json.load(f)
with open('../cleaned_data/kaggle_data.json', 'r') as f:
    kaggle_data = json.load(f)
with open('../cleaned_data/mendeley_cleaned.json', 'r') as f:
    mendeley_data = json.load(f)
print(f"BOW tokens: {len(kBOW)} for kaggle, {len(mBOW)} for mendeley")
print(f"Dataset entries: {len(kaggle_data)} for kaggle, {len(mendeley_data)} for mendeley")

Loading BOWs and datasets...
BOW tokens: 35466 for kaggle, 36601 for mendeley
Dataset entries: 117458 for kaggle, 53920 for mendeley


In [19]:
# ============ Combine BOWs ============
BOW = dict()
for count in mBOW:
    token = count['token']
    freq = count['count']
    BOW[token] = freq
for count in kBOW:
    token = count['token']
    freq = count['count']
    if token in BOW:
        BOW[token] += freq
    else:
        BOW[token] = freq
print(f"Combined BOW tokens: {len(BOW)}")


Combined BOW tokens: 54714


In [ ]:
# ============ Spell correct per token Kaggle ============
spell = autocorrect.Speller(lang="en", fast=True)
eng_stopwords = set(stopwords.words('english'))

def reduce_spammed(word: str) -> str:
    # if there are 3 or more repeated characters, reduce to 2
    return re.sub(r'(.)\1{2,}', r'\1\1', word)

k_corrected_map = dict()

for i, review in enumerate(kaggle_data):
    if i % 1000 == 0:
        print(f"Processing review {i}/{len(kaggle_data)}")
    tokens = word_tokenize(str(review["review_content"]).lower())
    for i, token in enumerate(tokens):
        token = reduce_spammed(token)
        tokens[i] = token
        # skip stopwords
        if token in eng_stopwords:
            continue
        # skip if BOW has seen the original token more than 10 times
        if token in BOW and BOW[token] > 10:
            continue
        corrected_token = spell(token)
        # skip if the corrected token is the same as the original token
        if corrected_token == token:
            continue
        # skip if BOW never seen the corrected or rarely seen it
        if corrected_token not in BOW or BOW[corrected_token] < 5:
            continue
        k_corrected_map[corrected_token] = token
        tokens[i] = corrected_token
    kaggle_data[i]["review_content"] = " ".join(tokens)

print(f"Corrected {len(k_corrected_map)} unique tokens in kaggle dataset")
print(f"Example corrections: {list(k_corrected_map.items())[:10]}")

Processing review 0/117458
Processing review 1000/117458
Processing review 2000/117458
Processing review 3000/117458
Processing review 4000/117458
Processing review 5000/117458
Processing review 6000/117458
Processing review 7000/117458
Processing review 8000/117458
Processing review 9000/117458
Processing review 10000/117458
Processing review 11000/117458
Processing review 12000/117458
Processing review 13000/117458
Processing review 14000/117458
Processing review 15000/117458
Processing review 16000/117458
Processing review 17000/117458
Processing review 18000/117458
Processing review 19000/117458
Processing review 20000/117458
Processing review 21000/117458
Processing review 22000/117458
Processing review 23000/117458
Processing review 24000/117458
Processing review 25000/117458
Processing review 26000/117458
Processing review 27000/117458
Processing review 28000/117458
Processing review 29000/117458
Processing review 30000/117458
Processing review 31000/117458
Processing review 320

In [27]:
# ============ Save corrected dataset kaggle ============
with open('../cleaned_data/kaggle_spellcorrected.json', 'w') as f:
    json.dump(kaggle_data, f, indent=4)

In [29]:
# ============ Spell correct per token Mendeley ============
m_corrected_map = dict()

for i, review in enumerate(mendeley_data):
    if i % 1000 == 0:
        print(f"Processing review {i}/{len(mendeley_data)}")
    tokens = word_tokenize(str(review["review"]).lower())
    for i, token in enumerate(tokens):
        token = reduce_spammed(token)
        tokens[i] = token
        # skip stopwords
        if token in eng_stopwords:
            continue
        # skip if BOW has seen the original token more than 10 times
        if token in BOW and BOW[token] > 10:
            continue
        corrected_token = spell(token)
        # skip if the corrected token is the same as the original token
        if corrected_token == token:
            continue
        # skip if BOW never seen the corrected or rarely seen it
        if corrected_token not in BOW or BOW[corrected_token] < 5:
            continue
        m_corrected_map[corrected_token] = token
        tokens[i] = corrected_token
    mendeley_data[i]["review"] = " ".join(tokens)

print(f"Corrected {len(m_corrected_map)} unique tokens in mendeley dataset")
print(f"Example corrections: {list(m_corrected_map.items())[:10]}")

Processing review 0/53920
Processing review 1000/53920
Processing review 2000/53920
Processing review 3000/53920
Processing review 4000/53920
Processing review 5000/53920
Processing review 6000/53920
Processing review 7000/53920
Processing review 8000/53920
Processing review 9000/53920
Processing review 10000/53920
Processing review 11000/53920
Processing review 12000/53920
Processing review 13000/53920
Processing review 14000/53920
Processing review 15000/53920
Processing review 16000/53920
Processing review 17000/53920
Processing review 18000/53920
Processing review 19000/53920
Processing review 20000/53920
Processing review 21000/53920
Processing review 22000/53920
Processing review 23000/53920
Processing review 24000/53920
Processing review 25000/53920
Processing review 26000/53920
Processing review 27000/53920
Processing review 28000/53920
Processing review 29000/53920
Processing review 30000/53920
Processing review 31000/53920
Processing review 32000/53920
Processing review 33000

In [30]:
# ============ Save corrected dataset mendeley ============
with open('../cleaned_data/mendeley_spellcorrected.json', 'w') as f:
    json.dump(mendeley_data, f, indent=4)